# Sofiv2 - Yıldız Katalogu Analiz Notebook'u

Bu notebook, tarihi ve modern yıldız kataloglarını yükleme, inceleme, görselleştirme ve cross-match işlemleri için interaktif bir ortam sağlar.

## İçindekiler
1. Ortam Kontrolü
2. Şablon Tablo Yükleme
3. Koordinat İşlemleri
4. Görselleştirme
5. Online Katalog Sorgusu (Astroquery)

## 1. Ortam Kontrolü
Tüm paketlerin kurulu olduğunu doğrulayalım.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from astropy.table import Table
from astropy.coordinates import SkyCoord
import astropy.units as u
from pathlib import Path

# Set plot style
sns.set_theme(style="darkgrid")
%matplotlib inline

ROOT = Path(".").resolve().parent
TEMPLATES = ROOT / "templates"

print(f"numpy {np.__version__}")
print(f"pandas {pd.__version__}")
print(f"Project root: {ROOT}")
print("\nOrtam hazir!")

## 2. Şablon Tablo Yükleme
CSV, FITS ve VOTable formatlarından yıldız katalog şablonunu okuyalım.

In [ ]:
# CSV ile pandas
df = pd.read_csv(TEMPLATES / "star_catalog_template.csv")
print(f"CSV: {len(df)} yildiz, {len(df.columns)} kolon")
df

In [ ]:
# FITS ile astropy
fits_table = Table.read(str(TEMPLATES / "star_catalog_template.fits"), format="fits")
print(f"FITS: {len(fits_table)} yildiz")
fits_table

In [ ]:
# VOTable ile astropy
vot_table = Table.read(str(TEMPLATES / "star_catalog_template.vot"), format="votable")
print(f"VOTable: {len(vot_table)} yildiz")
vot_table

## 3. Koordinat İşlemleri
Yıldız koordinatlarını astropy SkyCoord ile işleyelim.

In [ ]:
# SkyCoord nesneleri olustur
coords = SkyCoord(
    ra=df["RA_J2000"].values * u.degree,
    dec=df["Dec_J2000"].values * u.degree,
    frame="icrs"
)

# Galaktik koordinatlara donustur
galactic = coords.galactic

for i, name in enumerate(df["Star_Name"]):
    print(f"{name:12s}  RA={coords[i].ra.hms}  l={galactic[i].l:.2f}  b={galactic[i].b:.2f}")

In [ ]:
# Yildizlar arasi aci mesafeleri (separation matrix)
n = len(coords)
sep_matrix = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        sep_matrix[i][j] = coords[i].separation(coords[j]).degree

sep_df = pd.DataFrame(sep_matrix, index=df["Star_Name"], columns=df["Star_Name"])
print("Aci mesafe matrisi (derece):")
sep_df.round(1)

## 4. Görselleştirme

In [ ]:
# Gokyuzu haritasi (RA/Dec scatter plot)
fig, ax = plt.subplots(figsize=(12, 6))

scatter = ax.scatter(
    df["RA_J2000"], df["Dec_J2000"],
    s=(2 - df["Magnitude"]) * 100,  # Brighter = bigger
    c=df["Magnitude"],
    cmap="RdYlBu",
    edgecolors="black",
    linewidth=0.5,
    zorder=5
)

for _, row in df.iterrows():
    ax.annotate(
        row["Star_Name"],
        (row["RA_J2000"], row["Dec_J2000"]),
        textcoords="offset points",
        xytext=(8, 8),
        fontsize=10,
        fontweight="bold"
    )

plt.colorbar(scatter, label="Magnitude")
ax.set_xlabel("RA (J2000) [degree]")
ax.set_ylabel("Dec (J2000) [degree]")
ax.set_title("Sablon Katalog - Gokyuzu Haritasi")
ax.invert_xaxis()  # RA increases right-to-left
plt.tight_layout()
plt.show()

In [ ]:
# Parlaklik dagilimi (kataloglara gore)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
colors = sns.color_palette("husl", len(df))
axes[0].barh(df["Star_Name"], df["Magnitude"], color=colors)
axes[0].set_xlabel("Magnitude")
axes[0].set_title("Yildiz Parlakliklari")
axes[0].axvline(x=0, color="gray", linestyle="--", alpha=0.5)

# Separation heatmap
sns.heatmap(
    sep_df.round(1),
    annot=True,
    fmt=".1f",
    cmap="YlOrRd",
    ax=axes[1],
    square=True
)
axes[1].set_title("Aci Mesafe Matrisi (derece)")

plt.tight_layout()
plt.show()

## 5. Online Katalog Sorgusu (Astroquery)
SIMBAD'dan yıldız bilgisi çekelim. **Internet bağlantısı gerektirir.**

In [ ]:
from astroquery.simbad import Simbad

# Sirius hakkinda bilgi cek
result = Simbad.query_object("Sirius")
if result is not None:
    print("SIMBAD sorgusu basarili:")
    result.pprint()
else:
    print("Sorgu sonucu bos veya baglanti yok.")

In [ ]:
# Bir bolgedeki yildizlari sorgula (cone search)
from astroquery.simbad import Simbad

custom_simbad = Simbad()
custom_simbad.add_votable_fields("flux(V)")

# Sirius civarinda 0.5 derece yaricapta arama
center = SkyCoord(ra=101.2871 * u.degree, dec=-16.7161 * u.degree, frame="icrs")
result = custom_simbad.query_region(center, radius=0.5 * u.degree)

if result is not None:
    print(f"Sirius civari 0.5 derece icinde {len(result)} nesne bulundu:")
    result.pprint(max_lines=20)
else:
    print("Sonuc yok veya baglanti hatasi.")

---

## Sonraki Adimlar

- `Historical/` klasorune eski kataloglari ekle
- `Modern/` klasorune Gaia DR3 verisini indir (astroquery ile)
- Koordinat donusumu ve cross-match islemleri yap
- Sonuclari `Processed/` klasorune kaydet
- TOPCAT ile interaktif inceleme: `tools\topcat.bat`